# Preparacao dos Dados — Experimentos de Classificacao de Vulnerabilidades

**FrameworkPE · PPGES · UNIPAMPA**

Este notebook prepara os datasets de cada experimento a partir do dataset completo
de scans OpenVAS/Greenbone de containers Docker Hub (`openvas_experiments_dataset.csv`).

In [1]:
import pandas as pd
import uuid
from pathlib import Path

## 1. Carregar o dataset completo

In [1]:
df = pd.read_csv('openvas_experiments_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head(3)

NameError: name 'pd' is not defined

## 2. Exploracao dos dados

In [3]:
print('Distribuicao de Severity:')
print(df['Severity'].value_counts())
print(f'\nDistribuicao de Solution Type:')
print(df['Solution Type'].value_counts())

Distribuicao de Severity:
Severity
Medium      1887
Log         1675
High        1517
Critical    1027
Low          366
Name: count, dtype: int64

Distribuicao de Solution Type:
Solution Type
VendorFix        4233
Mitigation        428
WillNotFix        113
Workaround         66
NoneAvailable       7
Name: count, dtype: int64


In [4]:
print(f'Nomes de NVT unicos: {df["NVT Name"].nunique()}')
print(f'Result IDs unicos: {df["Result ID"].nunique()}')
print(f'Null em Vulnerability Insight: {df["Vulnerability Insight"].isna().sum()}')
print(f'Null em Impact: {df["Impact"].isna().sum()}')
print(f'Null em Solution: {df["Solution"].isna().sum()}')
print(f'Null em Affected Software/OS: {df["Affected Software/OS"].isna().sum()}')
print(f'Null em CVSS: {df["CVSS"].isna().sum()}')

Nomes de NVT unicos: 1285
Result IDs unicos: 6472
Null em Vulnerability Insight: 2036
Null em Impact: 4337
Null em Solution: 1624
Null em Affected Software/OS: 1995
Null em CVSS: 0


## 3. Experimento 1 — Classificacao de Severidade CVSS

**Input**: NVT Name + Summary + Vulnerability Insight + Impact (sem CVSS numerico)
**Target**: Severity (Informational / Low / Medium / High / Critical)

Seleciona as colunas textuais e renomeia `Severity` → `target`, `Result ID` → `id`.

In [5]:
COLUNAS_E1 = ['NVT Name', 'Summary', 'Vulnerability Insight', 'Impact', 'Severity', 'Result ID']

df_e1 = df[COLUNAS_E1].copy()
df_e1 = df_e1.rename(columns={'Severity': 'target', 'Result ID': 'id'})
print(f'Shape: {df_e1.shape}')
print(f'Targets: {df_e1["target"].value_counts().to_dict()}')
df_e1.head(3)

Shape: (6472, 6)
Targets: {'Medium': 1887, 'Log': 1675, 'High': 1517, 'Critical': 1027, 'Low': 366}


,NVT Name,Summary,Vulnerability Insight,Impact,target,id
0,PHP End of Life (EOL) Detection - Linux,The PHP version on the remote host has reached...,Each release branch of PHP is fully supported ...,An EOL version of PHP is not receiving any sec...,Critical,306b7c0f-b614-4fe8-93a0-7ab5645781eb
1,"PHP < 5.5.37, 5.6.x < 5.6.23 Multiple Vulnerab...",PHP is prone to multiple vulnerabilities.,The following flaws exist:\n\n - The 'spl_arr...,Successfully exploiting these issues allow rem...,Critical,6929e48a-9ac4-47e2-9b8f-91a4db3b05b9
2,PHP Multiple Vulnerabilities (Feb 2019) - Linux,PHP is prone to multiple vulnerabilities.,The following flaws exist:\n\n - Fixed bug #7...,NaN,Critical,54ae23cd-fbc7-4f8a-9d6c-60a6462d3030


In [6]:
OUTPUT_DIR = Path('experimento_1/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_e1.to_csv(OUTPUT_DIR / 'data_1.csv', index=False)
print(f'Salvo em {OUTPUT_DIR / "data_1.csv"}')

Salvo em experimento_1/data/data_1.csv


## 4. Experimento 2 — Classificacao do Tipo de Solucao

**Input**: NVT Name + Summary + Solution + Affected Software/OS
**Target**: Solution Type (VendorFix / WillNotFix / Workaround / Mitigation)

Seleciona as colunas e renomeia `Solution Type` → `target`, `Result ID` → `id`.

In [7]:
COLUNAS_E2 = ['NVT Name', 'Summary', 'Solution', 'Affected Software/OS', 'Solution Type', 'Result ID']

df_e2 = df[COLUNAS_E2].copy()
df_e2 = df_e2.rename(columns={'Solution Type': 'target', 'Result ID': 'id'})
print(f'Shape: {df_e2.shape}')
print(f'Targets: {df_e2["target"].value_counts().to_dict()}')
df_e2.head(3)

Shape: (6472, 6)
Targets: {'VendorFix': 4233, 'Mitigation': 428, 'WillNotFix': 113, 'Workaround': 66, 'NoneAvailable': 7}


,NVT Name,Summary,Solution,Affected Software/OS,target,id
0,PHP End of Life (EOL) Detection - Linux,The PHP version on the remote host has reached...,Update the PHP version on the remote host to a...,NaN,VendorFix,306b7c0f-b614-4fe8-93a0-7ab5645781eb
1,"PHP < 5.5.37, 5.6.x < 5.6.23 Multiple Vulnerab...",PHP is prone to multiple vulnerabilities.,"Update to version 5.5.37, 5.6.23 or later.",PHP prior to version 5.5.37 and 5.6.x prior to...,VendorFix,6929e48a-9ac4-47e2-9b8f-91a4db3b05b9
2,PHP Multiple Vulnerabilities (Feb 2019) - Linux,PHP is prone to multiple vulnerabilities.,"Update to version 5.6.40, 7.1.16, 7.2.14, 7.3....","PHP versions before 5.6.40, 7.x before 7.1.26,...",VendorFix,54ae23cd-fbc7-4f8a-9d6c-60a6462d3030


In [8]:
OUTPUT_DIR = Path('experimento_2/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_e2.to_csv(OUTPUT_DIR / 'data_2.csv', index=False)
print(f'Salvo em {OUTPUT_DIR / "data_2.csv"}')

Salvo em experimento_2/data/data_2.csv


### Variante 2.1 — Sem campo Solution (cenario zero-day)

Remove a coluna `Solution` do input.

In [9]:
COLUNAS_E2_1 = ['NVT Name', 'Summary', 'Affected Software/OS', 'Solution Type', 'Result ID']

df_e2_1 = df[COLUNAS_E2_1].copy()
df_e2_1 = df_e2_1.rename(columns={'Solution Type': 'target', 'Result ID': 'id'})
print(f'Shape: {df_e2_1.shape}')
df_e2_1.head(3)

Shape: (6472, 5)


,NVT Name,Summary,Affected Software/OS,target,id
0,PHP End of Life (EOL) Detection - Linux,The PHP version on the remote host has reached...,NaN,VendorFix,306b7c0f-b614-4fe8-93a0-7ab5645781eb
1,"PHP < 5.5.37, 5.6.x < 5.6.23 Multiple Vulnerab...",PHP is prone to multiple vulnerabilities.,PHP prior to version 5.5.37 and 5.6.x prior to...,VendorFix,6929e48a-9ac4-47e2-9b8f-91a4db3b05b9
2,PHP Multiple Vulnerabilities (Feb 2019) - Linux,PHP is prone to multiple vulnerabilities.,"PHP versions before 5.6.40, 7.x before 7.1.26,...",VendorFix,54ae23cd-fbc7-4f8a-9d6c-60a6462d3030


In [10]:
OUTPUT_DIR = Path('experimento_2_1/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_e2_1.to_csv(OUTPUT_DIR / 'data_2_1.csv', index=False)
print(f'Salvo em {OUTPUT_DIR / "data_2_1.csv"}')

Salvo em experimento_2_1/data/data_2_1.csv


## Resumo

| Experimento | Arquivo | Inputs | Target | Linhas |
|---|---|---|---|---|
| E1 | `data_1.csv` | NVT Name, Summary, Vulnerability Insight, Impact | Severity | 6472 |
| E2 | `data_2.csv` | NVT Name, Summary, Solution, Affected Software/OS | Solution Type | 6472 |
| E2.1 | `data_2_1.csv` | NVT Name, Summary, Affected Software/OS | Solution Type | 6472 |